<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementar do livro <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>, de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositório de código: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

<!-- aviso-traducao-ptbr -->
<sub>
<b>Tradução não oficial para português do Brasil.</b> Este arquivo é uma obra
derivada do repositório original de Sebastian Raschka
(<a href="https://github.com/rasbt/reasoning-from-scratch">rasbt/reasoning-from-scratch</a>),
licenciado sob Apache License 2.0. Apenas o texto foi traduzido; o código
permanece inalterado. Não é uma publicação oficial da Manning e não substitui o
livro. Detalhes das convenções em <code>GLOSSARIO-TRADUCAO.md</code>.
</sub>

# Capítulo 5: Soluções dos exercícios

Pacotes usados neste notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.22.2


&nbsp;
## Exercício 5.1: Usando o scorer heurístico como critério de desempate na self-consistency

- Há muitas formas de implementar isso
- Talvez a mais fácil seja tratar do lado de fora da função de self-consistency e trabalhar com o dicionário retornado (por exemplo, de forma parecida com o que fizemos no exercício 4.4, quando implementamos o desempate, que adicionamos diretamente à função `evaluate_math500_stream`)
- As linhas relevantes são mostradas abaixo

```python
# ...
from pathlib import Path
import time

from reasoning_from_scratch.ch05 import heuristic_score


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with heuristic_score
            else:
                best = None
                best_score = float("-inf")
            
                for cand in results["majority_winners"]:
                    scores = [
                        heuristic_score(results["full_answers"][idx], prompt=prompt)
                        for idx in results["groups"][cand]
                    ]
            
                    score = max(scores)
            
                    if score > best_score:
                        best_score = score
                        best = cand
            
                extracted = best

            # ...

    # ...
    return num_correct, num_examples, acc
```

- As melhorias em relação ao baseline do capítulo 3 e à self-consistency do capítulo 4 aparecem abaixo

|   | Método                                     | Modelo | Acurácia | Tempo     |
|---|--------------------------------------------|-------|----------|-----------|
| 1 | Baseline do capítulo 4 com prompting de CoT | Base  | 33,4%    | 129,2 min |
| 2 | Self-consistency (n=3) + voto majoritário  | Base  | 43,2%    | 328,2 min |
| 3 | Self-consistency (n=3) + heurística        | Base  | 43,4%    | 326,5 min |
| 4 | Self-consistency (n=3) + logprob médio     | Base  | 44,8%    | 327,7 min |

- Os valores de acurácia e os tempos de execução mostrados na tabela foram calculados sobre todas as 500 amostras do conjunto de teste MATH-500, usando uma GPU "cuda" (DGX Spark)

- Por conveniência, você pode rodar o script [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_consistency_scorer_math500.py), localizado em [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

- No entanto, note que, como discutido na [#159](https://github.com/rasbt/reasoning-from-scratch/issues/159), decidimos o vencedor da maioria com base no score heurístico, mas consideramos apenas a primeira ocorrência em cada par majoritário
- Por exemplo

&nbsp;
## Exercício 5.2: Usando o scorer heurístico em uma configuração best-of-N

- O best-of-N é parecido com a self-consistency no sentido de que geramos várias respostas
- No entanto, em vez de escolher a resposta final por voto majoritário, pontuamos todas as respostas com uma função de score (como `heuristic_score`) e retornamos a resposta de maior score
- Há várias formas de implementar esse comportamento, mas a mais fácil talvez seja usar a função de self-consistency já existente do capítulo 4 como modelo e encaixar nela o `heuristic_score`, como mostrado abaixo

```python
# ...

from reasoning_from_scratch.ch05 import (
    heuristic_score
)

def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

        score = heuristic_score(answer, prompt=prompt)

        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

```

- Os resultados são mostrados abaixo

|   | Método                                    | Modelo | Acurácia | Tempo     |
|---|-------------------------------------------|-------|----------|-----------|
| 1 | Baseline com prompting de chain-of-thought | Base  | 33,4%    | 129,2 min |
| 2 | Best-of-N (n=3) + heurística              | Base  | 40,6%    | 327,7 min |
| 3 | Best-of-N (n=3) + logprob médio           | Base  | 43,2%    | 330,2 min |

- Os valores de acurácia e os tempos de execução mostrados na tabela foram calculados sobre todas as 500 amostras do conjunto de teste MATH-500, usando uma GPU "cuda" (DGX Spark)

- Por conveniência, você pode rodar o script [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py), localizado em [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

&nbsp;
## Exercício 5.3: Usando o scorer de logprob como critério de desempate na self-consistency

- O código é parecido com o do exercício 5.1, exceto que trocamos `heuristic_score` por `avg_logprob_answer`

```python
# ...
# from reasoning_from_scratch.ch05 import heuristic_score
from reasoning_from_scratch.ch05 import avg_logprob_answer


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with avg_logprob_answer
            else:
                best = None
                best_score = float("-inf")
            
                # Consider all members of each majority group
                for cand in results["majority_winners"]:
                    scores = []
            
                    for idx in results["groups"][cand]:
                        candidate_full = results["full_answers"][idx]
            
                        score = avg_logprob_answer(
                            model=model,
                            tokenizer=tokenizer,
                            prompt=prompt,
                            answer=candidate_full,
                            device=device,
                        )
                        scores.append(score)
            
                    cand_score = max(scores)
            
                    if cand_score > best_score:
                        best_score = cand_score
                        best = cand
            
                extracted = best
            # ...

    # ...
    return num_correct, num_examples, acc
```

- As melhorias em relação ao baseline do capítulo 3 e à self-consistency do capítulo 4 aparecem abaixo

|   | Método                                    | Modelo | Acurácia | Tempo     |
|---|-------------------------------------------|-------|----------|-----------|
| 1 | Baseline com prompting de chain-of-thought | Base  | 33,4%    | 129,2 min |
| 2 | Self-consistency (n=3) + voto majoritário | Base  | 43,2%    | 328,2 min |
| 3 | Self-consistency (n=3) + heurística       | Base  | 43,4%    | 326,5 min |
| 4 | Self-consistency (n=3) + logprob médio    | Base  | 44,8%    | 327,7 min |

- Os valores de acurácia e os tempos de execução mostrados na tabela foram calculados sobre todas as 500 amostras do conjunto de teste MATH-500, usando uma GPU "cuda" (DGX Spark)

- Por conveniência, você pode rodar o script [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py), localizado em [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

&nbsp;
## Exercício 5.4: Usando o scorer de logprob em uma configuração best-of-N

- Para implementar o best-of-N com um scorer de logprob, podemos usar o código do exercício 5.2 e trocar o `heuristic_score` por `avg_logprob_answer`:

```python

from reasoning_from_scratch.ch05 import (
    avg_logprob_answer
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

            score = avg_logprob_answer(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                answer=answer,
                device=device
            )
        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }
```

- Os resultados são mostrados abaixo

| # | Método                                    | Modelo | Acurácia | Tempo     |
|---|-------------------------------------------|-------|----------|-----------|
| 1 | Baseline com prompting de chain-of-thought | Base  | 33,4%    | 129,2 min |
| 2 | Best-of-N (n=3) + heurística              | Base  | a definir | a definir |
| 3 | Best-of-N (n=3) + logprob médio           | Base  | a definir | a definir |

- Os valores de acurácia e os tempos de execução mostrados na tabela foram calculados sobre todas as 500 amostras do conjunto de teste MATH-500, usando uma GPU "cuda" (DGX Spark)

- Por conveniência, você pode rodar o script [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py), localizado em [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

&nbsp;
## Exercício 5.5: Usando o score heurístico para self-refinement

- Usar o `heuristic_score` é na verdade ainda mais simples do que usar o score de logprob; tudo o que precisamos fazer é mudar o seguinte código:

```python
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)


torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- O código atualizado fica:

```python
torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=heuristic_score,  # NEW
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- Os resultados, usando o scorer heurístico, aparecem nas linhas 4, 5 e 10:

|    | Método                 | Score         | Iterações  | Modelo     | Acurácia | Tempo     |
|----|------------------------|---------------|------------|------------|----------|-----------|
| 1  | Baseline (capítulo 3)  | -             | -          | Base       | 15,2%    | 10,1 min  |
| 2  | Self-refinement        | Nenhum        | 1          | Base       | 25,0%    | 84,8 min  |
| 3  | Self-refinement        | Nenhum        | 2          | Base       | 22,0%    | 165,4 min |
| 4  | Self-refinement        | Heurística    | 1          | Base       | 21,6%    | 84,7 min  |
| 5  | Self-refinement        | Heurística    | 2          | Base       | 20,8%    | 151,4 min |
| 6  | Self-refinement        | Logprob médio | 1          | Base       | 21,4%    | 85,3 min  |
| 7  | Self-refinement        | Logprob médio | 2          | Base       | 22,0%    | 165,3 min |
|    |                        |               |            |            |          |           |
| 8  | Baseline (capítulo 3)  | -             | -          | Reasoning  | 48,2%    | 182,1 min |
| 9  | Self-refinement        | Nenhum        | 1          | Reasoning  | 56,6%    | 498,8 min |
| 10 | Self-refinement        | Heurística    | 1          | Reasoning  | 57,8%    | 498,6 min |
| 11 | Self-refinement        | Logprob médio | 1          | Reasoning  | 48,4%    | 499,7 min |

- Os valores de acurácia e os tempos de execução mostrados na tabela foram calculados sobre todas as 500 amostras do conjunto de teste MATH-500, usando uma GPU "cuda" (DGX Spark)
- Por conveniência, você pode rodar o script [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_refinement_math500.py), localizado em [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)